# Upstream deforestation drainage-area map

This notebook builds a map for one municipality showing:

- the selected municipality outline
- all upstream drainage areas colored by long-difference deforestation
- all upstream river segments
- sensor stations attached to the upstream river network

Here deforestation is defined directly from `land_cover.feather` as normalized forest loss between `YEAR_START` and `YEAR_END` at the drainage-area level:

- `deforestation_count = forest_pixels_start - forest_pixels_end`
- `deforestation_share = deforestation_count / land_cover_total_start`

Expected inputs:

- `data/river_network/`
- `data/land_cover/land_cover.feather`
- `data/sensor_data/stations_rivers.parquet` or `data/sensor_data/sensor_data.duckdb`
- `data/gadm/gadm41_BRA.gpkg`


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import duckdb
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.append("../..")

from data.river_network import RiverNetwork

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 200)


In [ ]:
def locate_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "code").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root from the current working directory.")


PROJECT_ROOT = locate_project_root()
RIVER_NETWORK_DIR = PROJECT_ROOT / "data" / "river_network"
LAND_COVER_PATH = PROJECT_ROOT / "data" / "land_cover" / "land_cover.feather"
STATIONS_RIVERS_PATH = PROJECT_ROOT / "data" / "sensor_data" / "stations_rivers.parquet"
SENSOR_DB_PATH = PROJECT_ROOT / "data" / "sensor_data" / "sensor_data.duckdb"
MUNICIPALITY_PATH = PROJECT_ROOT / "data" / "gadm" / "gadm41_BRA.gpkg"
OUTPUT_FIGURE = PROJECT_ROOT / "output" / "figures" / "upstream_deforestation_municipality_map.png"

# Update these before running.
SELECTED_MUN_ID = "130260"
YEAR_START = 2000
YEAR_END = 2022
FOREST_CLASS_COLUMN = "land_cover_class_1"
TOTAL_AREA_COLUMN = "land_cover_total"
FIGSIZE = (12, 12)
BUFFER_FACTOR = 0.08

PROJECT_ROOT


In [ ]:
def require_path(path: Path, label: str) -> Path:
    if not path.exists():
        raise FileNotFoundError(f"Missing {label}: {path}")
    return path


def load_stations_rivers(parquet_path: Path, duckdb_path: Path) -> gpd.GeoDataFrame:
    if parquet_path.exists():
        stations = gpd.read_parquet(parquet_path)
        if stations.geometry.name != "geometry" and "geometry" in stations.columns:
            stations = stations.set_geometry("geometry")
        return stations.to_crs(4326)

    if duckdb_path.exists():
        with duckdb.connect(str(duckdb_path)) as connection:
            table_names = {row[0] for row in connection.execute("SHOW TABLES").fetchall()}
            if "stations_rivers" not in table_names:
                raise KeyError("Sensor database exists but does not contain a `stations_rivers` table.")
            frame = connection.execute("SELECT * FROM stations_rivers").fetchdf()
        if "geometry_wkt" not in frame.columns:
            raise KeyError("`stations_rivers` must contain `geometry_wkt`.")
        geometry = gpd.GeoSeries.from_wkt(frame["geometry_wkt"], crs=4326)
        return gpd.GeoDataFrame(frame, geometry=geometry, crs=4326)

    raise FileNotFoundError(
        "Could not find station geometry input. Expected either "
        f"{parquet_path} or {duckdb_path}."
    )


def load_municipalities(path: Path) -> gpd.GeoDataFrame:
    municipalities = gpd.read_file(require_path(path, "municipality boundaries"), layer="ADM_ADM_2").to_crs(4326)
    municipalities["adm2_id"] = municipalities["CC_2"].astype(str)
    if "CC_2r" in municipalities.columns:
        municipalities["mun_id"] = municipalities["CC_2r"].astype(str).str.zfill(6)
    elif "CC_2" in municipalities.columns:
        municipalities["mun_id"] = municipalities["CC_2"].astype(str).str.extract(r"(\d{6})", expand=False)
    else:
        municipalities["mun_id"] = pd.NA
    return municipalities


def resolve_selected_municipality(municipalities: gpd.GeoDataFrame, selected_mun_id: str) -> gpd.GeoDataFrame:
    selected_mun_id = str(selected_mun_id).zfill(6)
    selected = municipalities.loc[municipalities["mun_id"] == selected_mun_id].copy()
    if selected.empty:
        raise KeyError(
            "Selected municipality not found in the boundary file. "
            f"Expected `mun_id == {selected_mun_id}`."
        )
    return selected


def load_trench_adm2_matches(network: RiverNetwork, municipalities: gpd.GeoDataFrame) -> pd.DataFrame:
    trench_adm2 = getattr(network, "trench_adm2_table", None)
    if trench_adm2 is not None and not trench_adm2.empty:
        frame = trench_adm2[["trench_id", "adm2"]].dropna().copy()
        frame["adm2"] = frame["adm2"].astype(str)
        return frame.drop_duplicates(["trench_id", "adm2"])

    if network.trenches is None:
        raise ValueError("River network trenches are required to rebuild trench-to-ADM2 matches.")

    trenches = network.trenches[["trench_id", "geometry"]].dropna().copy()
    municipality_index = municipalities[["adm2_id", "geometry"]].rename(columns={"adm2_id": "adm2"})
    matches = gpd.sjoin(
        trenches.to_crs(4326),
        municipality_index.to_crs(4326),
        how="inner",
        predicate="intersects",
    )
    frame = matches[["trench_id", "adm2"]].copy()
    frame["adm2"] = frame["adm2"].astype(str)
    return frame.drop_duplicates(["trench_id", "adm2"])


def reachable_upstream_trenches(
    network: RiverNetwork,
    trench_adm2: pd.DataFrame,
    selected_adm2_ids: list[str],
) -> pd.DataFrame:
    selected_trench_ids = trench_adm2.loc[
        trench_adm2["adm2"].astype(str).isin(selected_adm2_ids),
        "trench_id",
    ].drop_duplicates().astype(np.int64)
    if selected_trench_ids.empty:
        raise ValueError("No river trenches were matched to the selected municipality.")

    rivers = network.trenches[["trench_id", "system_id", "trench_index"]].dropna().copy()
    rivers["trench_id"] = rivers["trench_id"].astype(np.int64)
    rivers["system_id"] = rivers["system_id"].astype(int)
    rivers["trench_index"] = rivers["trench_index"].astype(int)

    system_trench_tables = {
        int(system_id): system_rows[["trench_id", "trench_index"]]
        .sort_values("trench_index")
        .reset_index(drop=True)
        for system_id, system_rows in rivers.groupby("system_id")
    }
    system_trench_arrays = {
        system_id: system_rows["trench_id"].to_numpy(dtype=np.int64)
        for system_id, system_rows in system_trench_tables.items()
    }
    system_position_lookup = {
        system_id: dict(
            zip(
                system_rows["trench_id"].to_numpy(dtype=np.int64),
                system_rows["trench_index"].to_numpy(dtype=np.int64),
            )
        )
        for system_id, system_rows in system_trench_tables.items()
    }
    trench_lengths = (
        network.trenches[["trench_id", "distance"]]
        .drop_duplicates("trench_id")
        .assign(trench_id=lambda df: df["trench_id"].astype(np.int64))
        .rename(columns={"distance": "trench_length_km"})
    )

    frames: list[pd.DataFrame] = []
    selected_rows = rivers.loc[rivers["trench_id"].isin(selected_trench_ids)].copy()
    for system_id, seed_rows in selected_rows.groupby("system_id"):
        system_id = int(system_id)
        seed_positions = np.asarray(
            [
                system_position_lookup[system_id][int(trench_id)]
                for trench_id in seed_rows["trench_id"]
                if int(trench_id) in system_position_lookup[system_id]
            ],
            dtype=np.int64,
        )
        if len(seed_positions) == 0:
            continue

        system_reachability = network.trench_reachability_matrices[system_id][seed_positions, :].tocsr()
        system_distance = network.trench_distance_matrices[system_id][seed_positions, :].tocsr()
        system_trench_ids = system_trench_arrays[system_id]
        min_distances = np.full(len(system_trench_ids), np.inf)

        for row_idx in range(system_reachability.shape[0]):
            reach_start = system_reachability.indptr[row_idx]
            reach_end = system_reachability.indptr[row_idx + 1]
            reachable_cols = system_reachability.indices[reach_start:reach_end]
            if len(reachable_cols) == 0:
                continue

            dist_start = system_distance.indptr[row_idx]
            dist_end = system_distance.indptr[row_idx + 1]
            distance_cols = system_distance.indices[dist_start:dist_end]
            distance_vals = system_distance.data[dist_start:dist_end].astype(float, copy=False)
            if len(distance_cols) == len(reachable_cols) and np.array_equal(distance_cols, reachable_cols):
                reachable_distances = distance_vals
            else:
                distance_lookup = dict(zip(distance_cols.tolist(), distance_vals.tolist()))
                reachable_distances = np.asarray(
                    [distance_lookup.get(int(col_idx), 0.0) for col_idx in reachable_cols],
                    dtype=float,
                )
            np.minimum.at(min_distances, reachable_cols, reachable_distances)

        reachable_mask = np.isfinite(min_distances)
        if np.any(reachable_mask):
            frames.append(
                pd.DataFrame(
                    {
                        "trench_id": system_trench_ids[reachable_mask],
                        "upstream_distance_km": min_distances[reachable_mask],
                    }
                )
            )

    if not frames:
        raise ValueError("No upstream trench network was resolved for the selected municipality.")

    upstream = (
        pd.concat(frames, ignore_index=True)
        .groupby("trench_id", as_index=False)["upstream_distance_km"]
        .min()
        .merge(trench_lengths, on="trench_id", how="left", validate="one_to_one")
    )
    upstream["adjusted_distance_km"] = upstream["upstream_distance_km"] - upstream["trench_length_km"]
    return upstream.sort_values(["adjusted_distance_km", "trench_id"]).reset_index(drop=True)


def build_deforestation_long_difference(land_cover: pd.DataFrame, year_start: int, year_end: int) -> pd.DataFrame:
    required_columns = {"trench_id", "year", FOREST_CLASS_COLUMN, TOTAL_AREA_COLUMN}
    missing_columns = required_columns.difference(land_cover.columns)
    if missing_columns:
        raise KeyError(
            "Land-cover file is missing required column(s): "
            f"{sorted(missing_columns)}."
        )

    subset = land_cover[["trench_id", "year", FOREST_CLASS_COLUMN, TOTAL_AREA_COLUMN]].copy()
    subset["trench_id"] = pd.to_numeric(subset["trench_id"], errors="coerce").astype("Int64")
    subset = subset.dropna(subset=["trench_id"])
    subset["trench_id"] = subset["trench_id"].astype(np.int64)
    subset = subset.loc[subset["year"].isin([year_start, year_end])].copy()

    wide = subset.pivot_table(
        index="trench_id",
        columns="year",
        values=[FOREST_CLASS_COLUMN, TOTAL_AREA_COLUMN],
        aggfunc="sum",
    )

    for column in [FOREST_CLASS_COLUMN, TOTAL_AREA_COLUMN]:
        for year in [year_start, year_end]:
            key = (column, year)
            if key not in wide.columns:
                wide[key] = np.nan

    result = pd.DataFrame(index=wide.index)
    result["forest_pixels_start"] = wide[(FOREST_CLASS_COLUMN, year_start)]
    result["forest_pixels_end"] = wide[(FOREST_CLASS_COLUMN, year_end)]
    result["total_area_start"] = wide[(TOTAL_AREA_COLUMN, year_start)]
    result["deforestation_count"] = result["forest_pixels_start"] - result["forest_pixels_end"]
    result["deforestation_share"] = np.where(
        result["total_area_start"] > 0,
        result["deforestation_count"] / result["total_area_start"],
        np.nan,
    )
    return result.reset_index()


def buffered_bounds(geometries: gpd.GeoSeries, buffer_factor: float = 0.08) -> tuple[float, float, float, float]:
    xmin, ymin, xmax, ymax = geometries.total_bounds
    xpad = max((xmax - xmin) * buffer_factor, 0.05)
    ypad = max((ymax - ymin) * buffer_factor, 0.05)
    return xmin - xpad, xmax + xpad, ymin - ypad, ymax + ypad


In [ ]:
require_path(RIVER_NETWORK_DIR, "river network directory")
require_path(LAND_COVER_PATH, "trench-level land cover")
require_path(MUNICIPALITY_PATH, "municipality boundary file")

network = RiverNetwork()
network.load(str(RIVER_NETWORK_DIR))
if network.drainage_areas is None:
    raise ValueError("River network must include `drainage_areas.parquet` to plot upstream drainage polygons.")

municipalities = load_municipalities(MUNICIPALITY_PATH)
selected_municipality = resolve_selected_municipality(municipalities, SELECTED_MUN_ID)
selected_adm2_ids = selected_municipality["adm2_id"].astype(str).tolist()

trench_adm2 = load_trench_adm2_matches(network, municipalities)
upstream_trenches = reachable_upstream_trenches(network, trench_adm2, selected_adm2_ids)
upstream_trench_ids = upstream_trenches["trench_id"].astype(np.int64).tolist()

upstream_rivers = network.trenches.loc[
    network.trenches["trench_id"].astype(np.int64).isin(upstream_trench_ids)
].copy().to_crs(4326)

upstream_drainage_areas = network.drainage_areas.loc[
    network.drainage_areas["trench_id"].astype(np.int64).isin(upstream_trench_ids)
].copy().to_crs(4326)

land_cover = pd.read_feather(LAND_COVER_PATH)
deforestation_change = build_deforestation_long_difference(land_cover, YEAR_START, YEAR_END)
upstream_drainage_areas = upstream_drainage_areas.merge(
    deforestation_change,
    on="trench_id",
    how="left",
    validate="one_to_one",
)

stations = load_stations_rivers(STATIONS_RIVERS_PATH, SENSOR_DB_PATH)
stations["station_code"] = stations["station_code"].astype(str)
stations["trench_id"] = pd.to_numeric(stations["trench_id"], errors="coerce").astype("Int64")
upstream_stations = stations.loc[stations["trench_id"].isin(upstream_trenches["trench_id"])].copy()

summary = pd.DataFrame(
    {
        "metric": [
            "selected_adm2_rows",
            "upstream_drainage_areas",
            "upstream_trenches",
            "upstream_stations",
        ],
        "value": [
            len(selected_municipality),
            len(upstream_drainage_areas),
            len(upstream_trenches),
            len(upstream_stations),
        ],
    }
)
summary


In [ ]:
plot_geometries = pd.concat(
    [
        upstream_drainage_areas[["geometry"]],
        selected_municipality[["geometry"]],
        upstream_rivers[["geometry"]],
        upstream_stations[["geometry"]] if not upstream_stations.empty else gpd.GeoDataFrame(geometry=[], crs=4326),
    ],
    ignore_index=True,
)
plot_geometries = gpd.GeoDataFrame(plot_geometries, geometry="geometry", crs=4326)

xmin, xmax, ymin, ymax = buffered_bounds(plot_geometries.geometry, BUFFER_FACTOR)

fig, ax = plt.subplots(figsize=FIGSIZE)

upstream_drainage_areas.plot(
    ax=ax,
    column="deforestation_share",
    cmap="YlOrRd",
    linewidth=0.15,
    edgecolor="#636363",
    legend=True,
    legend_kwds={"label": f"Deforestation share, {YEAR_START} to {YEAR_END}", "shrink": 0.75},
    missing_kwds={"color": "#f3f3f3", "edgecolor": "#bdbdbd", "label": "No deforestation data"},
)

upstream_rivers.plot(ax=ax, color="#2c7fb8", linewidth=0.8, alpha=0.85, zorder=3)

if not upstream_stations.empty:
    upstream_stations.plot(
        ax=ax,
        color="#f28e2b",
        markersize=18,
        edgecolor="white",
        linewidth=0.4,
        zorder=4,
    )

selected_municipality.boundary.plot(ax=ax, color="black", linewidth=2.0, zorder=5)

ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
selected_label = selected_municipality.iloc[0].get("NAME_2", str(SELECTED_MUN_ID))
ax.set_title(
    f"Upstream drainage areas and deforestation share for {selected_label} ({SELECTED_MUN_ID})",
    fontsize=14,
)
ax.text(
    0.01,
    0.01,
    f"Drainage areas: {len(upstream_drainage_areas)} | Upstream river segments: {len(upstream_trenches)} | Stations: {len(upstream_stations)}",
    transform=ax.transAxes,
    fontsize=10,
    ha="left",
    va="bottom",
    bbox={"facecolor": "white", "alpha": 0.9, "edgecolor": "none"},
)
plt.show()


In [ ]:
OUTPUT_FIGURE.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(OUTPUT_FIGURE, dpi=300, bbox_inches="tight")
OUTPUT_FIGURE
